Task 4.1 — One validated call

In [1]:
!pip install -q google-genai pydantic

In [7]:
import json
import os
from google import genai
from google.genai import types
from getpass import getpass

# Define the schema as requested
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"]
}

# Updated to the recommended available Flash model
MODEL_NAME = 'gemini-3.5-flash-lite'

# Request API key securely
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")
client = genai.Client()

# Sample note N001 for test run
N001 = """Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week"""

def analyse_note(note_text):
    # CRAFT prompt parts
    system_instruction = "You are an expert data analyst. Analyze the provided patient response note to extract structured data."

    prompt = f"""
    Context: We are analyzing patient response notes to categorize them and assign actions.
    Role: Data Analyst
    Action: Analyze the following note, identify the root cause category, estimate your confidence level, summarize the issue in 25 words or less, and suggest a recommended action.
    Tone: Objective, analytical, concise
    Format: JSON matching the defined schema

    Input note:
    '{note_text}'
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text)

# Run on note N001 and display the output
result = analyse_note(N001)
import pprint
pprint.pprint(result)

Enter your Gemini API key: ··········
{'confidence': 0.95,
 'recommended_action': 'Reschedule the visit for next week and investigate the '
                       'reminder notification system.',
 'root_cause': 'Missed communication / Reminder failure',
 'summary': 'Patient missed the appointment due to not receiving a reminder '
            'message and requests to reschedule for next week.'}


Task 4.2 — Validate and process the 10 notes

In [ ]:
import time
import json
import pandas as pd
from pydantic import BaseModel, Field, field_validator, ValidationError

# Define the allowed list of root causes
ALLOWED_ROOT_CAUSES = {
    "administrative_error",
    "technical_issue",
    "communication_gap",
    "patient_preference",
    "other"
}

# Define the Pydantic model according to requirements
class FollowupInsight(BaseModel):
    root_cause: str = Field(..., min_length=3)
    confidence: float = Field(..., ge=0.0, le=1.0)
    summary: str = Field(..., min_length=5)
    recommended_action: str = Field(..., min_length=5)

    @field_validator('root_cause')
    @classmethod
    def check_allowed_root_cause(cls, v: str) -> str:
        if v not in ALLOWED_ROOT_CAUSES:
            raise ValueError(f"root_cause '{v}' must be one of {ALLOWED_ROOT_CAUSES}")
        return v

# Load the CSV dataset
df = pd.read_csv('/content/followup_notes.csv')

# Keep track of the results
results = []

# Process only the first 10 rows
for index, row in df.head(10).iterrows():
    note_id = row['note_id']
    appointment_id = row['appointment_id']
    note_text = row['note_text']

    print(f"Processing index {index} (Note ID: {note_id})...")

    # Input validation: check for missing, empty, or whitespace-only notes
    if pd.isna(note_text) or str(note_text).strip() == "":
        print("-> Skipping: Blank or empty note.")
        results.append({
            'note_id': note_id,
            'appointment_id': appointment_id,
            'root_cause': 'other',
            'confidence': 0.0,
            'summary': 'No note recorded',
            'recommended_action': 'Re-call patient',
            'validation_status': 'skipped_blank',
            'raw_response': ''
        })
        continue

    raw_json_text = ""
    try:
        # Call the existing analyse_note function
        # In case the API returns raw response objects, let's call it and capture its result
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=f"""
            Context: We are analyzing patient response notes to categorize them and assign actions.
            Role: Data Analyst
            Action: Analyze the following note, identify the root cause category, estimate your confidence level, summarize the issue in 25 words or less, and suggest a recommended action.
            Tone: Objective, analytical, concise
            Format: JSON matching the defined schema

            Input note:
            '{note_text}'
            """,
            config=types.GenerateContentConfig(
                system_instruction="You are an expert data analyst. Analyze the provided patient response note to extract structured data.",
                temperature=0,
                max_output_tokens=1000,
                response_mime_type="application/json",
                response_schema=SCHEMA
            )
        )

        raw_json_text = response.text
        parsed_data = json.loads(raw_json_text)

        # Pydantic validation
        insight = FollowupInsight(**parsed_data)
        validation_status = "valid"

        # Store details from valid response
        results.append({
            'note_id': note_id,
            'appointment_id': appointment_id,
            'root_cause': insight.root_cause,
            'confidence': insight.confidence,
            'summary': insight.summary,
            'recommended_action': insight.recommended_action,
            'validation_status': validation_status,
            'raw_response': raw_json_text
        })
        print("-> Valid response stored.")

    except Exception as e:
        print(f"-> Failed/Invalid response: {str(e)}")
        validation_status = "invalid"

        # Safely try to pull whatever keys existed in the parsed JSON
        try:
            partially_parsed = json.loads(raw_json_text) if raw_json_text else {}
        except:
            partially_parsed = {}

        results.append({
            'note_id': note_id,
            'appointment_id': appointment_id,
            'root_cause': partially_parsed.get('root_cause', None),
            'confidence': partially_parsed.get('confidence', None),
            'summary': partially_parsed.get('summary', None),
            'recommended_action': partially_parsed.get('recommended_action', None),
            'validation_status': validation_status,
            'raw_response': raw_json_text
        })

    # Pause to respect free tier rate limit of 4 RPM (or custom constraints)
    if index < 9: # Do not sleep after the last processed note
        print("Sleeping 13 seconds...")
        time.sleep(13)

# Save the results to CSV
output_df = pd.DataFrame(results)
output_df.to_csv('notes_analysed.csv', index=False)
print("Done! Saved to notes_analysed.csv")
display(output_df)

Processing index 0 (Note ID: N001)...
-> Failed/Invalid response: 1 validation error for FollowupInsight
root_cause
  Value error, root_cause 'Communication Failure' must be one of {'other', 'communication_gap', 'administrative_error', 'technical_issue', 'patient_preference'} [type=value_error, input_value='Communication Failure', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error
Sleeping 13 seconds...
Processing index 1 (Note ID: N002)...
-> Failed/Invalid response: 1 validation error for FollowupInsight
root_cause
  Value error, root_cause 'Transportation and Distance Barrier' must be one of {'other', 'communication_gap', 'administrative_error', 'technical_issue', 'patient_preference'} [type=value_error, input_value='Transportation and Distance Barrier', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error
Sleeping 13 seconds...
Processing index 2 (Note ID: N003)...
-> Failed/Invalid response: 

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,Communication Failure,0.95,Patient missed visit after not receiving a rem...,Reschedule appointment for next week and verif...,invalid,"{""root_cause"":""Communication Failure"",""confide..."
1,N002,A0897,Transportation and Distance Barrier,0.95,Patient was unable to attend appointment due t...,Offer telehealth options or provide transporta...,invalid,"{""root_cause"":""Transportation and Distance Bar..."
2,N003,A0526,Financial/Cost Concerns,0.95,Patient skipped their appointment due to high ...,Provide information regarding available financ...,invalid,"{""root_cause"":""Financial/Cost Concerns"",""confi..."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,Illness / Acute Symptoms,0.95,Patient missed visit due to high fever and bod...,Follow up with the patient in a few days to ch...,invalid,"{""root_cause"":""Illness / Acute Symptoms"",""conf..."
5,N006,A0507,Work Conflict,0.95,Patient missed appointment due to an urgent wo...,Reschedule the appointment to an available eve...,invalid,"{""root_cause"":""Work Conflict"",""confidence"":0.9..."
6,N007,A0333,Communication Failure / Missed Reminder,0.95,Patient forgot appointment because no SMS remi...,Assist patient in rescheduling appointment and...,invalid,"{""root_cause"":""Communication Failure / Missed ..."
7,N008,A0222,Transportation and Distance Barriers,0.95,Patient missed appointment due to a city bus s...,"Provide transportation assistance, offer trave...",invalid,"{""root_cause"":""Transportation and Distance Bar..."
8,N009,A0977,Patient Unreachable,0.95,Unable to reach the patient after three call a...,Attempt contact via alternative channels such ...,invalid,"{""root_cause"":""Patient Unreachable"",""confidenc..."
9,N010,A0596,Work Schedule Conflict,0.95,Patient experienced a factory shift change pre...,Reschedule the patient's appointment to an ava...,invalid,"{""root_cause"":""Work Schedule Conflict"",""confid..."


Task 4.3 — Break it on purpose

In [ ]:
import pandas as pd
import json
from pydantic import ValidationError

print("=== Part (a): Pydantic Model Validation Tests ===")

# Bad Reply 1: confidence = 1.7
bad_reply_1 = {
    "root_cause": "administrative_error",
    "confidence": 1.7,
    "summary": "Valid summary of the issue",
    "recommended_action": "Valid recommended action"
}

try:
    FollowupInsight(**bad_reply_1)
except ValidationError as e:
    print("Bad Reply 1 Captured Successfully!")
    print(e)
    print("-" * 50)

# Bad Reply 2: missing recommended_action
bad_reply_2 = {
    "root_cause": "technical_issue",
    "confidence": 0.8,
    "summary": "Valid summary of the issue"
}

try:
    FollowupInsight(**bad_reply_2)
except ValidationError as e:
    print("Bad Reply 2 Captured Successfully!")
    print(e)
    print("-" * 50)


print("\n=== Part (b): Prompt Comparison (V1 vs V2) ===")

# Extract notes N007 and N009 from the loaded DataFrame
note_007_text = df[df['note_id'] == 'N007']['note_text'].values[0]
note_009_text = df[df['note_id'] == 'N009']['note_text'].values[0]

test_cases = [
    ("N007", note_007_text),
    ("N009", note_009_text)
]

# Define the two prompt versions
prompt_v1 = lambda text: f"Classify this note:\n'{text}'"

prompt_v2 = lambda text: f"""
Context: We are analyzing patient response notes to categorize them and assign actions.
Role: Data Analyst
Action: Analyze the following note, identify the root cause category, estimate your confidence level, summarize the issue in 25 words or less, and suggest a recommended action.
Tone: Objective, analytical, concise
Format: JSON matching the defined schema
Allowed values for root_cause: {ALLOWED_ROOT_CAUSES}
If the note is highly ambiguous, or you cannot confidently assign one of the categories, assign "other" as the root_cause instead of guessing.

Input note:
'{text}'
"""

prompts = {
    "V1 (Simple)": prompt_v1,
    "V2 (CRAFT)": prompt_v2
}

comparison_results = []

for note_id, note_text in test_cases:
    for prompt_name, prompt_func in prompts.items():
        print(f"Executing {prompt_name} for {note_id}...")
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_func(note_text),
                config=types.GenerateContentConfig(
                    system_instruction="You are an expert data analyst. Analyze the provided patient response note to extract structured data",
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA
                )
            )

            parsed = json.loads(response.text)
            extracted_cause = parsed.get("root_cause", "None")
            is_valid = extracted_cause in ALLOWED_ROOT_CAUSES

            comparison_results.append({
                "Note ID": note_id,
                "Prompt Version": prompt_name,
                "Extracted Root Cause": extracted_cause,
                "Is in Allowed List?": is_valid,
                "Response Summary": parsed.get("summary", "None")
            })
        except Exception as e:
            print(f"API call failed for {note_id} with {prompt_name}: {e}")
            comparison_results.append({
                "Note ID": note_id,
                "Prompt Version": prompt_name,
                "Extracted Root Cause": "API_FAILED",
                "Is in Allowed List?": False,
                "Response Summary": str(e)
            })

# Show comparative results in a table
comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

print("\n=== Recommendation ===")
print("We recommend deploying Prompt V2 (CRAFT). It provides explicitly defined structural expectations, constraints for ambiguity (directing the model to fall back to 'other' rather than hallucinating), and references our allowed categories, yielding far more predictable and high-quality extractions.")

print("\n=== Part (c): API Call Audit ===")
# 1 call in Task 4.1, 9 calls in Task 4.2 (due to skipping 1 blank), and 4 calls in Task 4.3
total_api_calls = 1 + 9 + 4
print(f"Total number of API calls used across Tasks 4.1–4.3: {total_api_calls}")

=== Part (a): Pydantic Model Validation Tests ===
Bad Reply 1 Captured Successfully!
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
--------------------------------------------------
Bad Reply 2 Captured Successfully!
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'technical...d summary of the issue'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
--------------------------------------------------

=== Part (b): Prompt Comparison (V1 vs V2) ===
Executing V1 (Simple) for N007...
Executing V2 (CRAFT) for N007...
Executing V1 (Simple) for N009...
Executing V2 (CRAFT) for N009...


,Note ID,Prompt Version,Extracted Root Cause,Is in Allowed List?,Response Summary
0,N007,V1 (Simple),Did not receive SMS reminder / Forgot appointment,False,Patient missed the appointment because they fo...
1,N007,V2 (CRAFT),technical_issue,True,Patient missed appointment due to not receivin...
2,N009,V1 (Simple),Patient unreachable (phone switched off),False,Unable to contact the patient after three call...
3,N009,V2 (CRAFT),communication_gap,True,Unable to contact the patient after three atte...



=== Recommendation ===
We recommend deploying Prompt V2 (CRAFT). It provides explicitly defined structural expectations, constraints for ambiguity (directing the model to fall back to 'other' rather than hallucinating), and references our allowed categories, yielding far more predictable and high-quality extractions.

=== Part (c): API Call Audit ===
Total number of API calls used across Tasks 4.1–4.3: 14


In [ ]:
import pandas as pd

# Load analyzed notes and cleaned appointments
notes_df = pd.read_csv('notes_analysed.csv')
appointments_df = pd.read_csv('/content/appointments_clean.csv')

# Merge the datasets on appointment_id
merged_df = pd.merge(notes_df, appointments_df, on='appointment_id', how='inner')

# Perform cross-tabulation of root_cause against reminder_sent
cross_tab = pd.crosstab(
    index=merged_df['root_cause'].fillna('invalid/skipped'),
    columns=merged_df['reminder_sent'].fillna('unknown'),
    margins=True
)

print("=== Cross-tabulation: root_cause vs reminder_sent ===")
display(cross_tab)

=== Cross-tabulation: root_cause vs reminder_sent ===


reminder_sent,No,Yes,All
root_cause,,,
Communication Failure,1,0,1
Communication Failure / Missed Reminder,1,0,1
Financial/Cost Concerns,0,1,1
Illness / Acute Symptoms,0,1,1
Patient Unreachable,0,1,1
Transportation and Distance Barrier,1,0,1
Transportation and Distance Barriers,0,1,1
Work Conflict,0,1,1
Work Schedule Conflict,1,0,1
